In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F


import os, sys
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp

from collections import defaultdict

In [2]:
!pwd

/home/euna/hyperr


In [3]:
df = pd.read_csv('./data/tafeng.csv')

In [4]:
df

,TRANSACTION_DT,CUSTOMER_ID,AGE_GROUP,PIN_CODE,PRODUCT_SUBCLASS,PRODUCT_ID,AMOUNT,ASSET,SALES_PRICE
0,11/1/2000,1104905,45-49,115,110411,4710199010372,2,24,30
1,11/1/2000,418683,45-49,115,120107,4710857472535,1,48,46
2,11/1/2000,1057331,35-39,115,100407,4710043654103,2,142,166
3,11/1/2000,1849332,45-49,Others,120108,4710126092129,1,32,38
4,11/1/2000,1981995,50-54,115,100205,4710176021445,1,14,18
...,...,...,...,...,...,...,...,...,...
817736,2/28/2001,312790,35-39,114,530501,4713317035042,2,80,118
817737,2/28/2001,57486,40-44,115,530209,4710731060124,1,40,55
817738,2/28/2001,733526,>65,Unknown,510539,4716340052307,1,78,115
817739,2/28/2001,173704,45-49,115,520457,4714276145315,1,90,96


In [8]:
print(len(df['CUSTOMER_ID'].unique()))
print(len(df['PRODUCT_ID'].unique()))

32266
23812


In [6]:
df['TRANSACTION_DT'] = pd.to_datetime(df['TRANSACTION_DT'])
df = df[['TRANSACTION_DT', 'CUSTOMER_ID', 'PRODUCT_ID']]

In [7]:
df

,TRANSACTION_DT,CUSTOMER_ID,PRODUCT_ID
0,2000-11-01,1104905,4710199010372
1,2000-11-01,418683,4710857472535
2,2000-11-01,1057331,4710043654103
3,2000-11-01,1849332,4710126092129
4,2000-11-01,1981995,4710176021445
...,...,...,...
817736,2001-02-28,312790,4713317035042
817737,2001-02-28,57486,4710731060124
817738,2001-02-28,733526,4716340052307
817739,2001-02-28,173704,4714276145315


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 817741 entries, 0 to 817740
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   TRANSACTION_DT  817741 non-null  datetime64[ns]
 1   CUSTOMER_ID     817741 non-null  int64         
 2   PRODUCT_ID      817741 non-null  int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 18.7 MB


In [9]:
df.head(30)

,TRANSACTION_DT,CUSTOMER_ID,PRODUCT_ID
0,2000-11-01,1104905,4710199010372
1,2000-11-01,418683,4710857472535
2,2000-11-01,1057331,4710043654103
3,2000-11-01,1849332,4710126092129
4,2000-11-01,1981995,4710176021445
5,2000-11-01,1741797,78895770025
6,2000-11-01,308359,4710192225520
7,2000-11-01,1607000,4712936888817
8,2000-11-01,1057331,4715398106864
9,2000-11-01,236645,4710126091870


# 조건으로 유저, 아이템 지우기

In [10]:
while True:
    count_customer = df['CUSTOMER_ID'].value_counts()
    del_customer = count_customer[count_customer < 5].index
    lc = len(del_customer)
    
    if lc != 0:
        mask_customer = df['CUSTOMER_ID'].isin(del_customer)
        df = df[~mask_customer]
    


    count_product = df['PRODUCT_ID'].value_counts()
    del_product = count_product[count_product < 5].index
    lp = len(del_product)

    if lp != 0:
        mask_product = df['PRODUCT_ID'].isin(del_product)
        df = df[~mask_product] 
    elif lc == 0:
        break



In [12]:
df2 = df.groupby(['TRANSACTION_DT', 'CUSTOMER_ID'])['PRODUCT_ID'].apply(list).reset_index()
df2.rename(columns={'PRODUCT_ID' : 'bsk'}, inplace=True)
df2['product_count'] = df2['bsk'].apply(len)
df2

,TRANSACTION_DT,CUSTOMER_ID,bsk,product_count
0,2000-11-01,38317,"[4714981010038, 4719090105002]",2
1,2000-11-01,45902,"[4710147100018, 4710088434692, 4710594912028, ...",4
2,2000-11-01,45957,[4710265849066],1
3,2000-11-01,46855,"[4710088410139, 4710063031878, 4710085120468]",3
4,2000-11-01,58698,"[4710105010168, 4710105010182, 4710110910149, ...",6
...,...,...,...,...
111667,2001-02-28,2179490,"[4710168700044, 4710104111569, 4710561121125, ...",9
111668,2001-02-28,2179513,"[4014612509215, 4710320153121, 4710998000901, ...",70
111669,2001-02-28,2179544,"[20386214, 4710043355147, 20415457, 4712494216...",18
111670,2001-02-28,2179605,"[4710126001275, 4710475004057, 4710032601347, ...",52


In [12]:
df

,TRANSACTION_DT,CUSTOMER_ID,PRODUCT_ID,bsk,product_count
0,2000-11-01,1104905,4710199010372,"[4710199010372, 4711524000457, 4901201906015, ...",14
1,2000-11-01,418683,4710857472535,"[4710857472535, 4710011401128, 4710011401135, ...",4
2,2000-11-01,1057331,4710043654103,"[4710043654103, 4715398106864, 9310022530401, ...",4
3,2000-11-01,1849332,4710126092129,"[4710126092129, 4710011405133, 4710085120628, ...",28
4,2000-11-01,1981995,4710176021445,"[4710176021445, 4710543310066, 40000015321, 47...",6
...,...,...,...,...,...
784564,2001-02-28,312790,4713317035042,"[4710095937001, 4710254031359, 4710088414045, ...",8
784565,2001-02-28,57486,4710731060124,"[4710036220001, 4710105010182, 4710088434517, ...",11
784566,2001-02-28,733526,4716340052307,"[4714499311214, 8006330001948, 4710086190699, ...",6
784567,2001-02-28,173704,4714276145315,"[4714005052044, 4710291101039, 4714276145315]",3


In [ ]:
result_df['product_count'] = result_df['PRODUCT_ID'].apply(len)
result_df

,TRANSACTION_DT,CUSTOMER_ID,PRODUCT_ID,product_count
0,2000-11-01,38317,"[4714981010038, 4719090105002]",2
1,2000-11-01,45902,"[4710147100018, 4710088434692, 4710594912028, ...",4
2,2000-11-01,45957,[4710265849066],1
3,2000-11-01,46855,"[4710088410139, 4710063031878, 4710085120468]",3
4,2000-11-01,58698,"[4710105010168, 4710105010182, 4710110910149, ...",6
...,...,...,...,...
119573,2001-02-28,2179513,"[4014612509215, 4710320153121, 4710998000901, ...",70
119574,2001-02-28,2179544,"[20386214, 4710043355147, 20415457, 4712494216...",18
119575,2001-02-28,2179568,"[4710731010334, 4715222000566, 8999118015100, ...",5
119576,2001-02-28,2179605,"[4710126001275, 4710475004057, 4710032601347, ...",52


# 유저가 소비한 바구니들 list에 넣어주기 시간순으로

In [ ]:
result = []
tmp = []
for customer_id, group in result_df.sort_values(by='TRANSACTION_DT').groupby('CUSTOMER_ID'):
    # print('cu', customer_id)
    # print('group',group['PRODUCT_ID'])
    merged_list = [bsk for bsk in group['PRODUCT_ID']]
    # print('merged_list', merged_list)
    
    result.append([customer_id, merged_list])

result_df1 = pd.DataFrame(result, columns=['CUSTOMER_ID', 'PRODUCT_ID'])
result_df1

,CUSTOMER_ID,PRODUCT_ID
0,1069,"[[9556439880610, 4710176008699], [471032022466..."
1,1113,"[[4902105011621, 4711271000014], [490210501162..."
2,1250,"[[4719593555861, 4718585391258, 4905660070157,..."
3,1359,"[[4710088410139, 4710017008123, 5010415080073]]"
4,1823,"[[300410824006, 4710085172702, 300410824105, 4..."
...,...,...
32261,2179544,"[[20386214, 4710043355147, 20415457, 471249421..."
32262,2179568,"[[4710731010334, 4715222000566, 8999118015100,..."
32263,2179605,"[[4710126001275, 4710475004057, 4710032601347,..."
32264,2179643,"[[9310155100434, 4719111022202, 4710063312144,..."


In [ ]:
result_df1.loc[1069, 'PRODUCT_ID'][0]

[4710347906045,
 4970768741254,
 4710431038614,
 4710492811027,
 4902402376614,
 4897878000036,
 4710367814399,
 4902105113899,
 4904511001425,
 4710363502009,
 4710084225676,
 9310022724305,
 4710431052801,
 4710063413766,
 4710731070253,
 4711811101157,
 4710561201100,
 4710363502016,
 20459949,
 4711184739858,
 4713741200009,
 20398446,
 48001271326,
 4710049002052,
 4933193403913,
 4710718301011,
 4711290030092,
 4717265888293,
 4710265849066,
 4710126021204,
 4710431051668,
 4712566065534,
 9310022724008,
 9310022724206,
 4902430489065,
 4710431046824,
 4710154620264,
 4710088434593,
 4710363522007]

In [ ]:
import pandas as pd

# 주어진 데이터프레임 생성
data = {
    'TRANSACTION_DT': ['2000-11-01', '2000-11-01', '2000-11-01', '2000-11-01', '2000-11-01',
                       '2001-02-28', '2001-02-28', '2001-02-28', '2001-02-28', '2001-02-28'],
    'CUSTOMER_ID': [38317, 45902, 45957, 46855, 58698, 2179513, 2179544, 2179568, 2179605, 2179643],
    'PRODUCT_ID': [
        [4714981010038, 4719090105002],
        [4710147100018, 4710088434692, 4710594912028],
        [4710265849066],
        [4710088410139, 4710063031878, 4710085120468],
        [4710105010168, 4710105010182, 4710110910149],
        [4014612509215, 4710320153121, 4710998000901],
        [20386214, 4710043355147, 20415457, 4712494216121],
        [4710731010334, 4715222000566, 8999118015100],
        [4710126001275, 4710475004057, 4710032601347],
        [9310155100434, 4719111022202, 4710063312144]
    ]
}

df = pd.DataFrame(data)

# CUSTOMER_ID로 그룹화하고, PRODUCT_ID 리스트를 날짜 순으로 정렬하여 빈 리스트에 추가
result_df = (
    df.sort_values(by='TRANSACTION_DT')  # 날짜순으로 정렬
    .groupby('CUSTOMER_ID')['PRODUCT_ID']
    .agg(lambda x: [item for sublist in sorted(x) for item in sublist] if len(x) > 0 else [])
    .reset_index()
)

print(result_df)


   CUSTOMER_ID                                         PRODUCT_ID
0        38317                     [4714981010038, 4719090105002]
1        45902      [4710147100018, 4710088434692, 4710594912028]
2        45957                                    [4710265849066]
3        46855      [4710088410139, 4710063031878, 4710085120468]
4        58698      [4710105010168, 4710105010182, 4710110910149]
5      2179513      [4014612509215, 4710320153121, 4710998000901]
6      2179544  [20386214, 4710043355147, 20415457, 4712494216...
7      2179568      [4710731010334, 4715222000566, 8999118015100]
8      2179605      [4710126001275, 4710475004057, 4710032601347]
9      2179643      [9310155100434, 4719111022202, 4710063312144]


In [11]:
df = pd.read_csv('/home/euna/hyperr/data/my_tafeng.csv', index_col=0)

In [12]:
df

,user_id,bsks
0,0,"[[0, 41, 62, 80, 68, 135, 189, 221, 255, 284, ..."
1,1,"[[1, 35, 203, 173], [840], [564, 2390, 2063, 2..."
2,2,"[[2, 8, 130, 578], [1893, 1017], [1238], [608,..."
3,3,"[[3, 32, 33, 43, 81, 90, 101, 103, 112, 147, 1..."
4,4,"[[4, 55, 136, 68, 312, 580], [638], [81, 32, 3..."
...,...,...
9577,9577,"[[7077, 1485, 1364, 1456, 7577], [3451, 8199, ..."
9578,9578,"[[2262, 2719, 4013, 672, 3978], [4451, 2709, 1..."
9579,9579,"[[616, 4714, 7680], [3764, 8116, 1637, 36], [6..."
9580,9580,"[[4645, 8160, 4052, 8449, 1177, 8253, 8431, 83..."


In [2]:
# 주어진 리스트
original_list = [1, 2, 3, 4, 5]
# 제외할 인덱스
index_to_exclude = [1, 2]

# 특정 인덱스를 제외한 새 리스트 생성
new_list = [original_list[i] for i in range(len(original_list)) if i not in index_to_exclude]

# 결과 출력
print(new_list)


[1, 4, 5]
